In [0]:
import pandas as pd
import numpy as np

In [0]:
tnx=spark.table('workspace.default._fresh_mart_dataset_transactions')

transactions = tnx.toPandas()
display(transactions)

transaction_id,store_id,transaction_date,day_of_week,customer_id,is_loyalty_member,num_items,basket_value_zar,payment_method
TX0000001,ST1001,2025-01-01,Wednesday,null,No,25,487.08000000000000,Card
TX0000002,ST1001,2025-01-01,Wednesday,C889732,Yes,32,537.29000000000000,Cash
TX0000003,ST1001,2025-01-01,Wednesday,null,No,19,412.58000000000000,Cash
TX0000004,ST1001,2025-01-01,Wednesday,C955607,Yes,24,589.55000000000000,Card
TX0000005,ST1001,2025-01-01,Wednesday,C867011,Yes,1,19.98000000000000,Cash
TX0000006,ST1001,2025-01-01,Wednesday,null,No,20,333.32000000000000,Cash
TX0000007,ST1001,2025-01-01,Wednesday,C392524,Yes,27,406.10000000000000,Card
TX0000008,ST1001,2025-01-01,Wednesday,C873016,Yes,20,354.67000000000000,Cash
TX0000009,ST1001,2025-01-01,Wednesday,null,No,3,77.27000000000000,Cash
TX0000010,ST1002,2025-01-01,Wednesday,C944088,Yes,13,261.71000000000000,Mobile


In [0]:
transactions.shape

(64330, 9)

In [0]:
transactions.isnull().sum()

transaction_id           0
store_id                 0
transaction_date         0
day_of_week              0
customer_id          35674
is_loyalty_member        0
num_items                0
basket_value_zar         0
payment_method           0
dtype: int64

In [0]:
transactions["is_loyalty_member"].value_counts()

is_loyalty_member
No     35674
Yes    28656
Name: count, dtype: int64

In [0]:
transactions['payment_method'].value_counts()

payment_method
Card      32306
Cash      19754
Mobile    12270
Name: count, dtype: int64

In [0]:
transactions.describe()

,num_items
count,64330.000000
mean,13.448080
std,9.283258
min,1.000000
25%,4.000000
50%,15.000000
75%,21.000000
max,43.000000


In [0]:
transactions.duplicated().sum()

np.int64(0)

In [0]:
transactions.dtypes

transaction_id       object
store_id             object
transaction_date     object
day_of_week          object
customer_id          object
is_loyalty_member    object
num_items             int64
basket_value_zar     object
payment_method       object
dtype: object

In [0]:
#Converting basket_value_zar to float
transactions['basket_value_zar']= transactions['basket_value_zar'].astype(float).round(2)

In [0]:
display(transactions)

transaction_id,store_id,transaction_date,day_of_week,customer_id,is_loyalty_member,num_items,basket_value_zar,payment_method
TX0000001,ST1001,2025-01-01,Wednesday,null,No,25,487.08,Card
TX0000002,ST1001,2025-01-01,Wednesday,C889732,Yes,32,537.29,Cash
TX0000003,ST1001,2025-01-01,Wednesday,null,No,19,412.58,Cash
TX0000004,ST1001,2025-01-01,Wednesday,C955607,Yes,24,589.55,Card
TX0000005,ST1001,2025-01-01,Wednesday,C867011,Yes,1,19.98,Cash
TX0000006,ST1001,2025-01-01,Wednesday,null,No,20,333.32,Cash
TX0000007,ST1001,2025-01-01,Wednesday,C392524,Yes,27,406.1,Card
TX0000008,ST1001,2025-01-01,Wednesday,C873016,Yes,20,354.67,Cash
TX0000009,ST1001,2025-01-01,Wednesday,null,No,3,77.27,Cash
TX0000010,ST1002,2025-01-01,Wednesday,C944088,Yes,13,261.71,Mobile


In [0]:
#Converitng date to datetimes
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])

#Displaying results
print('Transaction date coverage:', transactions['transaction_date'].min().date(), 'to', transactions['transaction_date'].max().date())
print()
print('num_items summary:')
print(transactions['num_items'].describe())
print()
print('basket_value_zar summary:')
print(transactions['basket_value_zar'].describe())

Transaction date coverage: 2025-01-01 to 2025-12-31

num_items summary:
count    64330.000000
mean        13.448080
std          9.283258
min          1.000000
25%          4.000000
50%         15.000000
75%         21.000000
max         43.000000
Name: num_items, dtype: float64

basket_value_zar summary:
count                 64330
unique                38001
top       72.11000000000000
freq                     10
Name: basket_value_zar, dtype: object


In [0]:
#Confirms that every store_id in Transactions and Stockouts actually has a matching row in Stores

# Load stores and stockouts data
stores = spark.table('workspace.default.fresh_mart_stores').toPandas()
stockouts = spark.table('workspace.default.fresh_mart_stockouts').toPandas()

#Checking for stores that do not have a matching store
orphan_tx = transactions.loc[~transactions['store_id'].isin(stores['store_id']), 'store_id'].nunique()
orphan_so = stockouts.loc[~stockouts['store_id'].isin(stores['store_id']), 'store_id'].nunique()

#Display results
print(f"Distinct store_ids in Transactions: {transactions['store_id'].nunique()} (of {stores['store_id'].nunique()} stores in Stores table)")
print(f"Distinct store_ids in Stockouts:    {stockouts['store_id'].nunique()}")
print(f"Transaction store_ids not found in Stores table: {orphan_tx}")
print(f"Stockout store_ids not found in Stores table:    {orphan_so}")

Distinct store_ids in Transactions: 42 (of 42 stores in Stores table)
Distinct store_ids in Stockouts:    6
Transaction store_ids not found in Stores table: 0
Stockout store_ids not found in Stores table:    0


In [0]:
#is_loyalty_member and a blank customer_id should agree perfectly — this is checked explicitly rather than assumed.
pd.crosstab(transactions['is_loyalty_member'], transactions['customer_id'].isna(), rownames=['is_loyalty_member'], colnames=['customer_id is blank'])

customer_id is blank,False,True
is_loyalty_member,,
No,0,35674
Yes,28656,0


In [0]:
transactions['half'] = np.where(transactions['transaction_date'] < '2025-07-01', 'H1', 'H2')
transactions['month'] = transactions['transaction_date'].dt.to_period('M')

# Trip-type bands, based on the visible bimodal split in basket size (see Section 5)
transactions['trip_type'] = np.where(
    transactions['num_items'] <= 8, 'Small (top-up)',
    np.where(transactions['num_items'] <= 18, 'Medium', 'Large (stock-up)')
)

# Stockout-day flag: was there a recorded stockout at this store on this date?
stockout_days = stockouts[['store_id', 'date']].drop_duplicates()
stockout_days['date'] = pd.to_datetime(stockout_days['date'])
stockout_days['stockout_flag'] = 1
transactions = transactions.merge(
    stockout_days, left_on=['store_id', 'transaction_date'], right_on=['store_id', 'date'], how='left'
)
transactions['stockout_day'] = transactions['stockout_flag'].fillna(0).astype(int)
transactions = transactions.drop(columns=['date', 'stockout_flag'])

# Join store attributes onto every transaction for segmentation
df = transactions.merge(stores, on='store_id', how='left')

transactions[['transaction_id', 'half', 'month', 'trip_type', 'stockout_day']].head()

,transaction_id,half,month,trip_type,stockout_day
0,TX0000001,H1,2025-01,Large (stock-up),0
1,TX0000002,H1,2025-01,Large (stock-up),0
2,TX0000003,H1,2025-01,Large (stock-up),0
3,TX0000004,H1,2025-01,Large (stock-up),0
4,TX0000005,H1,2025-01,Small (top-up),0
